# 2. Open-Loop Control

[Previous lesson](01_dc_motor_dynamics.ipynb) · [Next lesson](03_proportional_control.ipynb)

A voltage command is not a speed guarantee. We will predict the unloaded response, choose voltage for a desired speed, and then apply a load without changing that voltage.

Work through the explanations, then implement each exercise. Hints and complete reference solutions are collapsed below your code cells. Solutions build in order; Run All does not execute them.

**Prerequisites:** complete the preceding lessons. This notebook starts independently: run the setup cell once before attempting the exercises.

## Previously implemented code

This setup repeats earlier material so you do not need another notebook running. Read it as a recap, then run it. The new work for this lesson remains in the empty cells below.

**Setup inventory:** `motor_dynamics` evaluates the two motor derivatives using SI units and state order `[current, omega]`. Where supplied, `simulate` advances the sampled controller and motor separately, and `plot_run` displays speed, voltage, and term contributions. Controller classes in setup were taught previously; their histories start fresh when you construct an instance. Run setup after restarting the kernel, then execute your exercise cells from top to bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

R, L, K_t, K_e, J, b = 1.0, 0.01, 0.1, 0.1, 0.01, 0.001

def motor_dynamics(t, state, voltage, load_torque):
    current, omega = state
    return [(voltage - R*current - K_e*omega)/L,
            (K_t*current - b*omega - load_torque)/J]


<img src="assets/lesson-02.svg" alt="Open-loop motor with independent load disturbance" style="display:block; width:100%; height:auto; margin:0 auto;">

<div class="figure-caption" style="background-color:rgba(128,128,128,0.10);padding:12px 18px;">

**Open-loop control.** External load enters the motor independently of voltage. No measurement returns to adjust the command.

</div>

## 2.1 From one experiment to a comparison

Open-loop control chooses an input without using the measured output to correct it. The motor still has internal electrical–mechanical coupling through back EMF; that physical effect is not an external speed controller.

Use identical starting conditions when comparing inputs. A larger voltage drives more current initially and, for our linear model, leads to a proportionally larger unloaded equilibrium speed. We keep the supply ideal and unlimited for now.

### What an open-loop experiment can tell us

Imagine a dial that sets motor voltage. Turning it changes electrical effort; it does not specify a speed that the motor must maintain. The speed emerges from the balance between current-produced torque, friction, and load. Even with no controller, back EMF changes as the rotor turns, so current naturally decreases during startup.

For this comparison, keep resistance, inductance, inertia, friction, initial state, duration, and output sampling unchanged. Otherwise a difference between curves could have several causes. Start every trial with zero current and speed: starting the next trial from the previous final state would answer a different question.

At the exact initial instant, current slope is $V/L$. Our 2, 4, and 6 V trials therefore start with current slopes of 200, 400, and 600 A/s. Acceleration is initially zero because current is initially zero, then rises as current creates torque. A higher-voltage curve should not start with an instantaneous nonzero speed.

The setup supplies `motor_dynamics(t, state, voltage, load_torque)` from Lesson 1. `solve_ivp` returns time in `result.t` and a two-row state array in `result.y`. Row zero is current; row one is speed. Plotting one row against time is the same operation as in the first notebook.

**Read the plot:** compare both the approach to equilibrium and the final plateau. The linear model predicts proportional final speeds, but it does not imply the motor jumps immediately to those speeds.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 1 — Compare fixed voltages

Simulate 2, 4, and 6 V for eight seconds from rest. Store the results and overlay the speed curves. Check solver success.

</div>

<details>
<summary>Show a hint</summary>

Use a dictionary keyed by voltage; each integration uses the same state and sample times.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
runs = {}
for voltage in (2.0, 4.0, 6.0):
    result = solve_ivp(motor_dynamics, (0,8), [0,0], args=(voltage,0.0),
                       t_eval=np.linspace(0,8,8001), rtol=1e-7, atol=1e-9)
    assert result.success
    runs[voltage] = result
    plt.plot(result.t, result.y[1], label=f'{voltage:g} V')
plt.xlabel('Time (s)'); plt.ylabel('Speed (rad/s)'); plt.legend(); plt.grid(alpha=.25)
plt.show()
```

All curves start at the same state. The plot changes only the applied voltage, making the comparison interpretable.

</details>

## 2.2 Include load in the equilibrium

At equilibrium, current supplies both viscous friction and external load torque:

$$K_t i_{ss}=b\omega_{ss}+\tau_L.$$

Substitute $i_{ss}=(b\omega_{ss}+\tau_L)/K_t$ into $V=Ri_{ss}+K_e\omega_{ss}$ and solve:

$$\omega_{ss}=\frac{K_tV-R\tau_L}{Rb+K_tK_e}.$$

The load term reduces the numerator. At 6 V, increasing load from zero to 0.1 N·m reduces the predicted speed from 54.545 to 45.455 rad/s. This formula is a signed equilibrium of the ideal model, not a guarantee that a physical motor behaves realistically under arbitrary loads.

### Derivation without skipping the substitution

At equilibrium, neither current nor speed changes. The electrical and mechanical equations become

$$V=Ri_{ss}+K_e\omega_{ss},\qquad K_ti_{ss}=b\omega_{ss}+\tau_L.$$

First solve the torque equation for current. Substitute that expression into the voltage equation:

$$V=\frac{Rb}{K_t}\omega_{ss}+\frac{R}{K_t}\tau_L+K_e\omega_{ss}.$$

Multiply by $K_t$ and collect all speed terms:

$$K_tV=(Rb+K_tK_e)\omega_{ss}+R\tau_L.$$

Moving the load term to the left gives the formula above. This is why the load appears with a minus sign: at fixed voltage, more electrical effort is spent sustaining load torque, leaving a lower equilibrium speed and back EMF.

For our constants, the denominator is $0.011$. At 6 V with 0.1 N·m load, the numerator is $0.6-0.1=0.5$, so speed is $0.5/0.011=45.455$ rad/s. The corresponding current is $(0.001\times45.455+0.1)/0.1=1.455$ A. Current is greater than in the unloaded case even though speed is lower.

An endpoint comparison is meaningful only after enough settling time. A formula mismatch while the speed is still changing is not necessarily an implementation error.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 2 — Validate loaded equilibrium

Implement the equilibrium formula. Compare each unloaded simulation endpoint with it, using 1% relative tolerance. Calculate the loaded prediction at 6 V and 0.1 N·m.

</div>

<details>
<summary>Show a hint</summary>

Keep voltage and load as function arguments.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
def equilibrium_speed(voltage, load=0.0):
    return (K_t*voltage-R*load)/(R*b+K_t*K_e)
for voltage, result in runs.items():
    np.testing.assert_allclose(result.y[1,-1], equilibrium_speed(voltage), rtol=.01)
print('Loaded speed:', equilibrium_speed(6.0,.1))
```

The algebra gives an independent target for validating the integrator.

</details>

## 2.3 Choose voltage for a desired speed

Rearrange the equilibrium equation to choose a nominal command:

$$V_{nom}=\frac{(Rb+K_tK_e)\omega_{target}+R\tau_{L,nom}}{K_t}.$$

For 40 rad/s and zero assumed external load, the required voltage is 4.4 V. This is a model-based feedforward calculation: it uses an assumed load rather than correcting measured error.

### A numerical feedforward design

For a desired 40 rad/s with zero assumed load, torque balance requires $i=0.001(40)/0.1=0.4$ A. Back EMF is 4 V and the resistive drop is 0.4 V, so the required supply is 4.4 V. This two-step physical calculation agrees with the compact inverse formula.

Now keep 4.4 V but add 0.1 N·m. The numerator of the speed formula becomes $0.44-0.1=0.34$, giving 30.909 rad/s. The target is still 40; the command has not changed. Nothing in this experiment compares actual speed with target, so nothing explicitly corrects the difference.

Feedforward is not inherently bad. If the load were known, including it in the inverse calculation would request 5.4 V and recover the nominal 40 rad/s equilibrium. The limitation is reliance on the model and assumed operating conditions. Feedback will allow correction when those assumptions are imperfect.

Keep the words **target**, **prediction**, and **measurement** distinct. The target is what we want, the formula predicts what the model will do, and the simulation supplies a time history. They need not coincide.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 3 — Choose feedforward voltage

Calculate the unloaded voltage for 40 rad/s and predict the speed when an unmodeled 0.1 N·m load is present.

</div>

<details>
<summary>Show a hint</summary>

Use the inverse formula, then call equilibrium_speed with the actual load.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
target = 40.0
nominal_voltage = (R*b+K_t*K_e)*target/K_t
loaded_speed = equilibrium_speed(nominal_voltage,.1)
print(nominal_voltage, loaded_speed)
np.testing.assert_allclose(nominal_voltage,4.4)
assert loaded_speed < target
```

The predicted loaded speed is about 30.909 rad/s, despite the voltage being correct for the unloaded case.

</details>

## 2.4 Apply a load step

A load disturbance changes the plant while the command stays fixed. Integrate the unloaded system first, then use its final state as the initial state for the loaded segment. Splitting at the discontinuity prevents the adaptive solver from stepping over an abrupt change without resolving it.

The state remains continuous at the event. Acceleration changes immediately, but speed does not jump.

<div class="lesson-callout" style="border-left: 3px solid #b99a62; background-color: rgba(185, 154, 98, 0.08); padding: 12px 18px; margin: 16px 0;">

**Key distinction**

A constant voltage cannot generally maintain constant speed when the load changes. The missing ingredient is correction based on measured error.

</div>

### Interpret the event at eight seconds

Immediately before the load step, the motor is near equilibrium. Current cannot jump because of inductance, and speed cannot jump because of inertia. With both states initially unchanged, the added 0.1 N·m makes acceleration change by $-0.1/0.01=-10$ rad/s².

As speed falls, back EMF falls. The electrical equation then permits current to rise. More current produces more motor torque, and eventually the system settles at a lower speed where the new torque balance holds. This is a chain of coupled responses, not an instantaneous speed drop.

The two integration segments meet at the same state. Including the boundary sample in both plots is harmless, but if you concatenate arrays for later numerical analysis, omit one duplicate timestamp. A zero time difference would make a numerical derivative invalid.

Add a vertical marker at the disturbance time and keep a horizontal target line. The vertical marker explains the trigger; the horizontal line makes the persistent error visible. Do not rename the loaded plateau as a new target: the experiment deliberately leaves the target unchanged.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 4 — Apply a load step

Run at the nominal voltage for eight seconds, apply 0.1 N·m, then continue to 16 seconds. Plot target and speed; check the final speed against the loaded prediction.

</div>

<details>
<summary>Show a hint</summary>

Pass the first segment’s final state into the second segment.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
before = solve_ivp(motor_dynamics,(0,8),[0,0],args=(nominal_voltage,0.0),
                   t_eval=np.linspace(0,8,2001),rtol=1e-7,atol=1e-9)
after = solve_ivp(motor_dynamics,(8,16),before.y[:,-1],args=(nominal_voltage,.1),
                  t_eval=np.linspace(8,16,2001),rtol=1e-7,atol=1e-9)
assert before.success and after.success
np.testing.assert_allclose(after.y[1,-1],loaded_speed,rtol=.01)
plt.plot(before.t,before.y[1]); plt.plot(after.t,after.y[1])
plt.axhline(target,ls='--',label='Target'); plt.axvline(8,color='gray',ls=':')
plt.xlabel('Time (s)'); plt.ylabel('Speed (rad/s)'); plt.legend(); plt.show()
```

At the load step, torque balance is disturbed and speed falls. The fixed voltage provides no explicit correction back to the original target.

</details>

## What the experiment establishes

You have separated a voltage command from a speed objective. The equilibrium model predicts what happens under a specified load, but a wrong load assumption produces a lasting speed error. The next lesson will measure that error and use it to adjust voltage.

## Before moving on

Run your exercises from a fresh kernel in order. Check the stated numerical conditions and explain your plots in your own words: what changed, why it changed, and which model assumptions limit the conclusion. Reference solutions are one implementation, not the only correct approach.

[Previous lesson](01_dc_motor_dynamics.ipynb) · [Next lesson](03_proportional_control.ipynb)

## References and further reading

[SciPy solve_ivp reference](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html) — callback arguments, output arrays, sample times, and tolerances.